In [ ]:
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.IamDataFrame("raw/REMIND-MAgPIE-RESCUE-Tier1-2025-12-08.xlsx")

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                "RESCUE-" + i[20:].replace("Budg", "-Budget-")
                .replace("Eoc", "End-of-Century")
                .replace("Pk", "Peak")
                .replace("-OAE_off", "")
                .replace("-OAE_on", "-with-OAE")
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df

In [ ]:
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/kW/yr": "USD_2010/kW/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "EJ/billion US$2010": "EJ/billion USD_2010",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "million m3/yr": "km3/yr",
        "bn m2/yr": "billion m2",
        "bn tkm/yr": "billion tkm/yr",
        "bn pkm/yr": "billion pkm/yr",
        "US$2010/GJ": "USD_2010/GJ",
        "Mt/year": "Mt/yr",
        "kt CF4-equiv/yr": "kt CF4/yr", 
    },
    inplace=True,
)

In [ ]:
project = ["navigate", "ngfs5"]
legacy_mapping = {}

for code, attrs in definition.variable.items():
    if project in attrs.extra_attributes:
        legacy_mapping[attrs.__getattr__(project)] = code

df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
df.rename(variable={
    "Food|Availability": "Food|Availability [per capita]",
    "Carbon Sequestration|Direct Air Capture": "Carbon Capture|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
}, inplace=True)

In [ ]:
df.rename(
    variable=dict(
        [
            (i, i.replace("Final Energy|Industry|Steel", "Final Energy|Industry|Iron and Steel"))
            for i in df.variable if i.startswith("Final Energy|Industry|Steel")
        ]
    ), 
    inplace=True,
)

In [ ]:
df.rename(
    variable=dict(
        [
            (i, i.replace("Final Energy|Industry|Cement", "Final Energy|Industry|Non-Metallic Minerals|Cement"))
            for i in df.variable if i.startswith("Final Energy|Industry|Cement")
        ]
    ), 
    inplace=True,
    )

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
# this would need to be reindexed to 2020=1
df.filter(variable="Price|Agriculture|Food Products*", keep=False, inplace=True)

In [ ]:
definition.validate(df, dimensions=["variable"])

In [ ]:
df.set_meta(meta="RESCUE [Horizon Europe]", name="Project")
df.set_meta(meta="n/a", name="Scientific Manuscript (DOI)")
df.set_meta(meta="Merfort et al. (2025)", name="Scientific Manuscript (Citation)")

In [ ]:
processor = nomenclature.RegionProcessor.from_directory(
    "../../common-definitions/mappings/",
    dsd=definition,
)

In [ ]:
df = processor.apply(df)

In [ ]:
import ixmp4

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
df.to_ixmp4(platform)